##### **Processing prev_data**

In [1]:
def fetch_json_data(file_path, output_file_path="cust_data.json"):
    """
    Fetches recipe data from a local JSON file, processes it, and returns only the relevant fields.
    """
    try:
        with open(file_path, 'r') as file:
            data = json.load(file)

        all_recipes = []

        # Iterate over each entry in the JSON file
        for entry in data:
            # Skip if entry is not a dictionary
            if not isinstance(entry, dict):
                print(f"Skipping non-dictionary entry: {entry}")
                continue

            recipe_info_list = entry.get("recipe_info", [])  # Ensure it's a list

            # Ensure recipe_info_list is actually a list before proceeding
            if not isinstance(recipe_info_list, list):
                print(f"Skipping entry due to unexpected format: {entry}")
                continue

            # Extract delivery date - now much simpler
            delivery_date = None
            entry_id = entry.get("_id", {})
            if isinstance(entry_id, dict):
                delivery_date = entry_id.get("delivery_date")
                # You might want to validate the date format here if needed

            # Iterate through each recipe_info dictionary in the list
            for recipe_info in recipe_info_list:
                # Skip if recipe_info is not a dictionary
                if not isinstance(recipe_info, dict):
                    print(f"Skipping non-dictionary recipe_info: {recipe_info}")
                    continue

                # Extract ingredients and merge with variant ingredients
                ingredients = recipe_info.get("ingredients", []) or []  # Ensure it's always a list
                variant_ingredients = []

                # Process the variants if available
                variants = recipe_info.get("variants", {})
                if isinstance(variants, dict):  # Ensure "variants" is a dictionary
                    variant_ingredients = variants.get("variant_ingredients", []) or []  # Ensure it's always a list

                # Merge ingredients from both the recipe_info and variants
                all_ingredients = list(set(ingredients + variant_ingredients))

                # Prepare the recipe data with only the necessary fields
                recipe = {
                    "dish_name": recipe_info.get("dish_name"),
                    "meal_category": recipe_info.get("meal_category"),
                    "description": recipe_info.get("description"),
                    "cuisine": recipe_info.get("cuisine"),
                    "ingredients": all_ingredients,  # Merged ingredients
                    "allergens_contain": recipe_info.get("allergens_contain", []),
                    "meal_type": entry.get("meal_type"),
                    "spice_level": recipe_info.get("spice_level", ""),
                    "is_auto_select": recipe_info.get("is_auto_select"),
                    "rating": recipe_info.get("rating") if "rating" in recipe_info else None,
                    "delivery_date": delivery_date  # Use the date string directly
                }

                # Add the processed recipe data to the list
                all_recipes.append(recipe)

        # If an output file path is provided, save the processed data to that file
        if output_file_path and all_recipes:
            with open(output_file_path, 'w') as output_file:
                json.dump(all_recipes, output_file, indent=4)
            print(f"Processed data saved to {output_file_path}")

        return all_recipes

    except FileNotFoundError:
        print(f"Error: The file '{file_path}' was not found.")
        return []
    except json.JSONDecodeError:
        print(f"Error: Failed to decode JSON data in file '{file_path}'. Please check the JSON syntax.")
        return []
    except Exception as e:
        print(f"Error processing file {file_path}: {str(e)}")
        return []

In [3]:
import os
import json
# Define the input and output directories
input_directory = "user_data/prev_data"
output_directory = "user_data/processed_files"

# Ensure output directory exists
os.makedirs(output_directory, exist_ok=True)

# Get all JSON files in the input directory
input_file_paths = [os.path.join(input_directory, file) for file in os.listdir(input_directory) if file.endswith(".json")]

# Loop through each input file and process it
for input_file_path in input_file_paths:
    file_name = os.path.basename(input_file_path)
    output_file_path = os.path.join(output_directory, f"processed_{file_name}")
    
    processed_data = fetch_json_data(input_file_path, output_file_path)
    
    if processed_data:
        print(f"Processed data for {file_name} saved to {output_file_path}")
    else:
        print(f"Failed to process data for {file_name}.")


Processed data saved to user_data/processed_files/processed_67b2e9941b9346c2cc4fcb4b.json
Processed data for 67b2e9941b9346c2cc4fcb4b.json saved to user_data/processed_files/processed_67b2e9941b9346c2cc4fcb4b.json
Processed data saved to user_data/processed_files/processed_659af0c214c28bbc7574d932.json
Processed data for 659af0c214c28bbc7574d932.json saved to user_data/processed_files/processed_659af0c214c28bbc7574d932.json
Processed data saved to user_data/processed_files/processed_67c87e618a4937be42c89279.json
Processed data for 67c87e618a4937be42c89279.json saved to user_data/processed_files/processed_67c87e618a4937be42c89279.json
Processed data saved to user_data/processed_files/processed_67b0ca24bb439e10a7e42e23.json
Processed data for 67b0ca24bb439e10a7e42e23.json saved to user_data/processed_files/processed_67b0ca24bb439e10a7e42e23.json
Processed data saved to user_data/processed_files/processed_66e0a84abd17c8733adb739f.json
Processed data for 66e0a84abd17c8733adb739f.json saved

##### **Pinecone query generation**

In [4]:
import pandas as pd

import json
from IPython.display import display, Markdown

def process_and_analyze_json(output_file_path="processed_data.json"):
    """
    Process and analyze the recipe data from a JSON file.
    It will:
    - Extract and merge the ingredients.
    - Analyze top cuisines, spice levels, and user selections.
    - Display visualizations like pie charts.
    - List all user-rated recipes sorted from highest to lowest rating.
    """
    try:
        # Load the processed JSON data directly
        with open(output_file_path, "r") as file:
            recipes = json.load(file)

        # Convert the processed data into a DataFrame
        df = pd.DataFrame(recipes)

        # Check if 'cuisine' column exists
        if 'cuisine' not in df.columns:
            raise KeyError("Missing 'cuisine' column in the dataset")

        # Identify the top cuisine preference
        top_cuisine = df['cuisine'].mode()
        # print(f"Top Cuisine: {top_cuisine}")
        
        # Identify the user's most preferred spice level
        top_spice_level = df['spice_level'].mode()[0]
        # print(f"Top Spice Level: {top_spice_level}")

        # No longer filtering by `is_auto_select`
        user_selected_meals = df.copy()  

        # Display a summary of all meals
        user_selected_meals_summary = user_selected_meals.describe(include='object')
        # print("Summary of All Selected Meals:")
        # display(user_selected_meals_summary)

        # Get the most frequent dish names
        top_dish_names = user_selected_meals['dish_name'].value_counts().reset_index()
        top_n = 5
        # display(Markdown("### Most Frequently Selected Dishes:"))
        # display(top_dish_names.head(top_n))

        # Get the count of each cuisine selected by the user
        cuisine_counts = user_selected_meals['cuisine'].value_counts().reset_index()
        cuisine_counts.columns = ["Cuisine", "Count"]
        # display(Markdown("### Total Cuisine Counts Across all Weeks :"))
        # display(cuisine_counts)

        # Display all user-rated recipes sorted from highest to lowest rating
        if 'rating' in df.columns:
            top_rated_recipes = df[df['rating'].notna()].sort_values(by='rating', ascending=False)
            # display(Markdown("### User-Rated Recipes (Sorted by Rating):"))
            # display(top_rated_recipes[['dish_name', 'rating', 'cuisine', 'spice_level', 'delivery_date']])

        if 'rating' in df.columns:
            least_rated_recipes = df[df['rating'].notna()].sort_values(by='rating', ascending=True)
            # display(Markdown("### User-Rated Recipes (Sorted by Rating):"))
            # display(least_rated_recipes[['dish_name', 'rating', 'cuisine', 'spice_level', 'delivery_date']])

        # Pie Chart for Cuisine Preferences
        # sns.set(style="darkgrid")
        # plt.style.use('dark_background')

        # plt.figure(figsize=(4, 4))  
        # plt.pie(
        #     cuisine_counts['Count'], 
        #     labels=cuisine_counts['Cuisine'], 
        #     autopct='%1.1f%%',  
        #     colors=sns.color_palette("tab20", len(cuisine_counts)), 
        #     startangle=90, 
        #     wedgeprops={'edgecolor': 'none'},  
        #     labeldistance=1.1,  
        #     pctdistance=0.85  
        # )
        # plt.title('Cuisines Selected by User', fontsize=8, color='white')  
        # plt.ylabel('')
        # plt.gcf().patch.set_alpha(0)
        # # display(Markdown("### Cuisine Preference Across all Weeks :"))
        # plt.show()

        # Weekly Cuisine Preferences Analysis
        if 'delivery_date' in df.columns:
            df['delivery_date'] = df['delivery_date'].apply(lambda x: str(x) if isinstance(x, dict) else x)
            df['delivery_date'] = pd.to_datetime(df['delivery_date'], errors='coerce')
        
        min_date = df['delivery_date'].min()
        df['week_number'] = df['delivery_date'].apply(lambda x: (x - min_date).days // 7 + 1)

        cuisine_weekly_counts = df.groupby(['week_number', 'cuisine']).size().reset_index(name='count')
        pivot_table = cuisine_weekly_counts.pivot(index="week_number", columns="cuisine", values="count").fillna(0)
        cuisine_order = cuisine_weekly_counts.groupby("cuisine")["count"].sum().sort_values()
        pivot_table = pivot_table[cuisine_order.index]

        # display(Markdown("### Cuisine Preferences over Weeks :"))
        # display(pivot_table)

        # Generate structured query based on analysis
        top_cuisines = ', '.join(df['cuisine'].mode())  # Get top cuisine(s)
        top_spice_level = df['spice_level'].mode()[0]  # Get top spice level
        # Ensure the first column contains dish names
        top_dishes = ', '.join(top_dish_names.iloc[:6, 0])  # ✅ Corrected way

        # Find highest-rated dishes (handling multiple)
        if 'rating' in df.columns and not df['rating'].isna().all():
            max_rating = df['rating'].max()  # Get the highest rating
            top_rated_dishes = df[df['rating'] == max_rating]['dish_name'].unique()  # Get unique top-rated dishes
            top_rated_dishes = ', '.join(top_rated_dishes[:5])  # Limit to top 5 for readability
        else:
            top_rated_dishes = None
        # Find dishes rated less than 3 (User Dislikes)
        user_dislikes = None
        if 'rating' in df.columns and not df['rating'].isna().all():
            disliked_dishes = df[df['rating'] < 3]['dish_name'].unique()  # Get unique low-rated dishes
            user_dislikes = ', '.join(disliked_dishes[:5])  # Limit to 5 for readability

        # Build the query string
        query = f"Spice Level: {top_spice_level}, Cuisine: {top_cuisines}. Popular Dishes: {top_dishes}"

        if top_rated_dishes:
            query += f". Highest Rated Dishes: {top_rated_dishes}"

        # print("Generated Query:")
        # print(query)

        return df, query, user_dislikes  # Return DataFrame and Query


    except FileNotFoundError:
        print(f"Error: The file '{output_file_path}' was not found.")
        return []
    except KeyError as e:
        print(f"Error: {str(e)}")
        return []
    except json.JSONDecodeError:
        print(f"Error: Failed to decode JSON data in file '{output_file_path}'. Please check the JSON syntax.")
        return []
    except Exception as e:
        print(f"Error: {str(e)}")
        return []

In [5]:
import os
import glob

# Folder containing the JSON files
folder_path = "user_data/processed_files/"

# Get all JSON files in the folder
json_files = glob.glob(os.path.join(folder_path, "*.json"))

# Process each file
for file_path in json_files:
    print(f"\nProcessing: {file_path}")
    df, query, user_dislikes = process_and_analyze_json(file_path)  # Call function for each file
    print(f"Query for {file_path}:\n{query}\n")



Processing: user_data/processed_files/processed_66476957f69e6aba7d9ade81.json
Query for user_data/processed_files/processed_66476957f69e6aba7d9ade81.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Balkan Mushroom Rice, Mac & Cheese with Cauliflower, Saloona with Green Beans , Thai Mango Salad, Maftoul with Zucchini & Minced Protein , Leek & Potato Fusilli Pasta. Highest Rated Dishes: Creamy Protein & Mash Potatoes, Mansaf , Fusilli Alfredo, Mac & Cheese with Cauliflower, Chipotle Lime Protein with Cauliflower Pilaf 


Processing: user_data/processed_files/processed_64dde7c55d497c8a38be700d.json
Query for user_data/processed_files/processed_64dde7c55d497c8a38be700d.json:
Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Burrito Bowl, Lentil Curry & Almond Black Rice, Korean Style Wrap, Baked Protein & Mashed Potato, Quinoa with Ratatouille , Tikka Protein with Saffron Rice


Processing: user_data/processed_files/processed_651dae63bed63f9897fec329.json
Quer

### **CSV collection**

In [7]:
import os
import json
import pandas as pd
import glob

def clean_string(value):
    """Helper function to clean strings and strip whitespace"""
    if isinstance(value, str):
        return value.strip()
    return value

def clean_string_list(value_list):
    """Helper function to clean lists of strings and strip whitespace"""
    if isinstance(value_list, (list, set)):
        return [item.strip() for item in value_list if isinstance(item, str)]
    return value_list

def extract_user_preferences(csv_path="Auto_Selection_Data.csv", json_folder="user_data/processed_files/"):
    """
    Extracts user preferences from CSV and JSON files and returns a structured dictionary.
    Trims whitespace from all string fields and lists.
    """
    # Load CSV
    df_csv = pd.read_csv(csv_path)

    # Dictionary to store user preferences
    user_preferences = {}

    # Get all JSON files in the folder
    json_files = glob.glob(os.path.join(json_folder, "*.json"))

    # Process each matching file
    for file_path in json_files:
        # Extract user_id from file name and clean it
        user_id = clean_string(os.path.basename(file_path).replace("processed_", "").replace(".json", ""))

        # Find matching row in CSV
        matching_row = df_csv[df_csv['customer_id'] == user_id]

        if not matching_row.empty:
            print(f"\nProcessing: {file_path}")

            # Extract values from CSV row
            row = matching_row.iloc[0]

            # Avoid ingredients (default to empty set if missing)
            user_avoid_ingredients = set(
                clean_string(ingredient) 
                for ingredient in str(row["avoid_ingredient"]).split(',') 
                if pd.notna(row["avoid_ingredient"])
            ) if pd.notna(row["avoid_ingredient"]) else set()

            # Protein category (default to empty string if missing)
            protein_category = clean_string(row['protein_category']) if pd.notna(row['protein_category']) else ""

            # Size (default to empty string if missing)
            size = clean_string(row["variant_size"]) if pd.notna(row["variant_size"]) else ""

            # Meal type as a set (default to empty set if missing)
            meal_types = set(
                clean_string(meal_type) 
                for meal_type in str(row["plan"]).split(',') 
                if pd.notna(row["plan"])
            ) if pd.notna(row["plan"]) else set()

            # Process JSON file
            df, query, user_dislikes = process_and_analyze_json(file_path)
            
            # Clean the query results
            cleaned_query = clean_string(query)
            user_dislikes = clean_string_list(user_dislikes)

            # Store results in a dictionary with cleaned values
            user_preferences[user_id] = {
                "user_pref": clean_string(cleaned_query.split("Popular Dishes: ")[1]) 
                           if "Popular Dishes: " in cleaned_query else "",
                "user_likes": f"{clean_string(cleaned_query.split('Cuisine: ')[1].split('.')[0])} cuisine" 
                             if "Cuisine: " in cleaned_query else "",
                "user_avoid_ingredients": user_avoid_ingredients,
                "size": size,
                "protein_option": "",  # Always default to empty string
                "protein_category": protein_category,
                "meal_types": meal_types,
                "query": cleaned_query,
                "user_dislikes": user_dislikes
            }

    return user_preferences

### **Recommendations**

In [10]:
import os
from dotenv import load_dotenv
from langchain.vectorstores import Pinecone as LangChainPinecone
from langchain.embeddings import HuggingFaceEmbeddings
from pinecone import Pinecone
from langchain_google_genai import ChatGoogleGenerativeAI
import google.generativeai as genai
import json
# from recipe_filter import filter_allergens_in_variants, filter_and_sort_recipes
import ast

# Load environment variables
load_dotenv(override=True)

gemini_api_key = os.getenv('GOOGLE_API_KEY')

# Ensure your Google API key is set
genai.configure(api_key=gemini_api_key)

# Initialize Pinecone
pc = Pinecone(api_key=os.getenv('PINECONE_API_KEY'))
index_name = "recipes-fin"

# Load the embedding model (same as used for storing data)
embed_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Connect Pinecone to LangChain
vectorstore = LangChainPinecone(pc.Index(index_name), embed_model, text_key="text")

# Initialize ChatGoogleGenerativeAI for gemini-1.5-flash
llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

# Function to generate responses
def generate_response(prompt):
    model = genai.GenerativeModel("gemini-1.5-flash")
    # Generate content based on the prompt
    response = model.generate_content(prompt)
    return response.text

# Function to filter and sort recipes
def filter_recipes(vectorstore, user_avoid_ingredients, user_dislikes, query, meal_category, size, protein_option, protein_category, top_k):
    pinecone_filter = {
        "meal_category": {"$eq": meal_category},  # Filter for specific meal type
        "size": {"$eq": size},  # Filter for specific size
        # "protein_option": {"$eq": protein_option},  # Filter for specific protein option
        "protein_category": {"$eq": protein_category},  # Filter for specific protein category
        "allergens": {"$nin": list(user_avoid_ingredients)},  # Exclude recipes containing allergens
        "ingredients": {"$nin": list(user_avoid_ingredients)}  # Exclude recipes containing allergens
    }
    # # Add `protein_category` filter only if it's not empty
    # if protein_category:
    #     pinecone_filter["protein_category"] = {"$eq": protein_category}


    # Fetch documents using similarity_search
    docs = vectorstore.similarity_search(
        query=query,
        k=top_k * 3 ,  # Get extra to account for duplicates
        filter=pinecone_filter
    )

    # Merge recipes by recipe_id and combine protein_options
    merged_recipes = {}

    
    for doc in docs:
        metadata = doc.metadata
        recipe_id = metadata.get("recipe_id")
        
        # Skip if no recipe_id exists
        if not recipe_id:
            continue
        
        # Initialize new entry if recipe_id not seen
        if recipe_id not in merged_recipes:
            merged_recipes[recipe_id] = {
                **metadata,  # Copy all metadata
                "protein_option": {metadata.get("protein_option")}  # Start as set
            }
        else:
            # Merge protein_options
            existing = merged_recipes[recipe_id]
            new_protein = metadata.get("protein_option")
            if new_protein:
                existing["protein_option"].add(new_protein)
    
    # Prepare final output (convert sets to lists)
    final_recipes = [
        {
            **data,
            "protein_option": list(data["protein_option"]) if data["protein_option"] else []
        }
        for data in merged_recipes.values()
    ]
    
    return final_recipes[:top_k]

# Function to format the filtered recipes into a structured meal plan prompt
def format_meal_plan_prompt(merged_recipes, query, user_avoid_ingredients, user_likes, user_dislikes, user_pref, meal_types):

        # Normalize meal types by stripping whitespace and converting to lowercase
    normalized_meal_types = {meal.strip().lower() for meal in meal_types}
    
    # Define the standard meal type order (customize as needed)
    STANDARD_ORDER = ['morning_snack', 'breakfast', 'lunch', 'dinner', 'evening_snack']
    
    # Filter and order the meal types based on standard order
    ordered_meal_types = [meal for meal in STANDARD_ORDER 
                         if meal in normalized_meal_types]
    
    prompt = f"""Generate a weekly meal plan in JSON format using ONLY the provided recipes. Follow these rules exactly:

1. Recipe Usage:
- Use recipes exactly as provided - do not modify or create new ones
- Format each meal as: "<Dish Name> - <Selected Protein> - <Cuisine> - <Dish Type>"
- Use ONLY the dish_type field for the last component (never meal_category)
- For recipes with multiple protein options:
    * Ensure protein variety across the week (don't serve chicken 3 days in a row)

3. Meal Diversity:
- Alternate between:
  * Light vs heavy meals (e.g. salad → hearty stew)
  * Different cuisines (don't repeat back-to-back)
  * Cooking methods (grilled, baked, fried, etc.)
- Ensure no two consecutive meals have:
  * The same primary ingredient
  * Similar textures/flavor profiles

2. Meal Assignment:
- Never repeat recipes before all are used once
- Fill all selected meal slots - no empty values
- **Strictly follow meal categories:**
    * Breakfast: only 'breakfast' recipes having meal_category as breakfast
    * Lunch/Dinner: only 'meal' recipes having meal_category as meal
    * evening_snavk/morning_snack: only 'snack' recipes i.e. recipes having meal_category as snack
- Include only these meal types: {meal_types}

3. Daily Structure:
- You MUST include these meal types in EXACTLY this order: {ordered_meal_types}
- Never skip or rearrange these meal types
- Never include meal types not in this list
- Maintain consistent meal types across all days

Output Format: Present the meal plan as a JSON object where each day contains meal types as keys and the formatted meal string as values, like this example for Monday: {{\"Monday\": {{\"breakfast\": \"Dish Name - Protein - Cuisine - Dish Type\", \"lunch\": \"...\"}}}}User Preferences:
- Allergens: {user_avoid_ingredients}
- Likes: {user_likes}
- Dislikes: {user_dislikes}
- Preferred Dishes: {user_pref}
- Selected Meal Types: {meal_types}

Available Recipes:"""
    

    for i, recipe in enumerate(merged_recipes, 1):
        prompt += f"Meal {i}:\n"
        prompt += f" Dish Name: {recipe.get('dish_name', 'Unknown')}\n"
        prompt += f" Description: {recipe.get('description', 'No description')}\n"
        prompt += f" Protein Options: {', '.join(recipe.get('protein_option', []))}\n"  # Changed to handle list
        prompt += f" Dish Type: {', '.join(recipe.get('dish_type', []))}\n"  # Changed to handle list
        prompt += f" Ingredients: {', '.join(recipe.get('ingredients', []))}\n"
        prompt += f" Spice Level: {recipe.get('spice_level', 'Not specified')}\n"
        prompt += f" Cuisine: {recipe.get('cuisine', 'Unknown')}\n"
        prompt += f" Meal Category: {recipe.get('meal_category', 'Unknown')}\n"
    return prompt

# Main function to generate the meal plan
def generate_meal_plan(vectorstore, user_avoid_ingredients, user_dislikes, query, user_likes, user_pref, size, protein_option, protein_category, meal_types):
    # Define mapping of meal types to their respective counts
    meal_counts = {
        "breakfast": 8,
        "snack": 8,  # Each snack type (morning/evening) adds 8
        "meal": 0  # Lunch and Dinner are combined into "meal"
    }

    # Initialize recipe counts
    total_snack_count = 0
    total_meal_count = 0  # To hold combined lunch + dinner count
    fetched_recipes = {}

    # Determine the total number of snack recipes needed
    if "morning_snack" in meal_types or "evening_snack" in meal_types:
        total_snack_count = meal_counts["snack"] * sum(1 for meal in meal_types if "snack" in meal)

    # If lunch or dinner is included, sum their counts into "meal"
    if "lunch" in meal_types:
        total_meal_count += 10
    if "dinner" in meal_types:
        total_meal_count += 10

    # Fetch recipes for each meal type
    for meal_type in meal_types:
        # if "snack" in meal_type or meal_type in ["lunch", "dinner"]:
        #     continue  # Skip individual snacks and meals; handle separately

        count = meal_counts.get(meal_type, 0)
        if count > 0:
            # Set size to "standard" for breakfast and snack
            meal_size = "standard" if meal_type in ["breakfast", "snack"] else size

            fetched_recipes[meal_type] = filter_recipes(
                vectorstore, user_avoid_ingredients, user_dislikes, query, meal_type, meal_size, "", protein_category, count
            )

    # Fetch total meal recipes (combined lunch & dinner) with user-specified size
    if total_meal_count > 0:
        fetched_recipes["meal"] = filter_recipes(
            vectorstore, user_avoid_ingredients, user_dislikes, query, "meal", size, protein_option, protein_category, total_meal_count
        )

    # Fetch total snack recipes (combined morning & evening snacks) with "standard" size
    if total_snack_count > 0:
        fetched_recipes["snack"] = filter_recipes(
            vectorstore, user_avoid_ingredients, user_dislikes, query, "snack", "standard", "", protein_category, total_snack_count
        )

    # Debugging: Print fetched recipe counts
    # for meal_type, recipes in fetched_recipes.items():
    #     print(f"{meal_type.capitalize()} recipes: {len(recipes)}")
    

    # Check if the total number of recipes is less than expected
    total_recipes = sum(len(recipes) for recipes in fetched_recipes.values())
    # expected_total = sum(meal_counts.get(meal, 0) for meal in meal_types if meal != "snack") + total_snack_count + total_meal_count
    # expected_total = count + total_snack_count + total_meal_count
    expected_total = (
    max(0, count - 4) +
    max(0, total_snack_count - (3 if total_snack_count == 8 else 6 if total_snack_count == 16 else 0)) +
    max(0, total_meal_count - (2 if total_meal_count == 8 else 7 if total_meal_count == 20 else 0))
    )

    if total_recipes < expected_total:
        print("Not enough recipes found. Retrying without allergens filter.")
        user_avoid_ingredients = {}  # Reset allergens and retry

        if total_meal_count > 0:
            fetched_recipes["meal"] = filter_recipes(
                vectorstore, user_avoid_ingredients, user_dislikes, query, "meal", size, protein_option, protein_category, total_meal_count
            )

        if total_snack_count > 0:
            fetched_recipes["snack"] = filter_recipes(
                vectorstore, user_avoid_ingredients, user_dislikes, query, "snack", "standard", "", protein_category, total_snack_count
            )

    # Print fetched recipe counts for each meal type
    print("Fetched recipes per meal type:")
    for meal_type, recipes in fetched_recipes.items():
        print(f"{meal_type.capitalize()}: {len(recipes)}")

    
    
    # Combine all selected recipes into the final list
    final_docs = [recipe for recipes in fetched_recipes.values() for recipe in recipes]
   
    #  # Step 3: Format the structured meal plan prompt
    final_prompt = format_meal_plan_prompt(final_docs, query, user_avoid_ingredients, user_likes, user_dislikes, user_pref, meal_types)

    # # Step 4: Generate the response using LLM
    meal_plan = generate_response(final_prompt)

    # print(meal_plan)
    return meal_plan, final_docs
    # return final_docs, final_prompt

# Example usage
if __name__ == "__main__":
    # User preferences (replace with dynamic input if needed)
    user_avoid_ingredients = {}  # Example allergens
    # user_dislikes = {
    #     "Cod (white fish)", "Cod Fish", "Cuttlefish", "Fish Sauce",
    #     "Gochujang Paste", "Gochujang Sauce", "Local Wild Fish",
    #     "Nile Perch", "Salmon", "Sea Bass", "Squid", "Tuna",
    #     "White Fish", "Worcestershire Sauce"
    # }  # Example disliked ingredients
    
    # query = "Spice Level: Medium, Cuisine: Mediterranean,European,Comfort Food. Popular Dishes: Classic Chicken Salad, Crudites & Sour Cream Dip, Mini Quiches, Omega Egg Protein Pot"

    # Generate the meal plan
    # generate_meal_plan(vectorstore, user_avoid_ingredients, user_dislikes, query, user_likes, user_pref)

In [13]:
import time

# Call function to extract user preferences
user_preferences = extract_user_preferences()

# Iterate over all users and generate meal plans
for user_id, prefs in user_preferences.items():
    print(f"\nGenerating meal plan for User ID: {user_id}")

    # Extract user details
    user_avoid_ingredients = prefs["user_avoid_ingredients"] if prefs["user_avoid_ingredients"] else {}  
    user_dislikes = prefs["user_dislikes"] if prefs["user_dislikes"] else {}  
    query = prefs["query"]
    user_likes = prefs["user_likes"]
    user_pref = prefs["user_pref"]
    size = prefs["size"].lower() if prefs["size"] else ""  # Convert to lowercase
    protein_option = prefs["protein_option"] if prefs["protein_option"] else ""  
    protein_category = prefs["protein_category"].lower() if prefs["protein_category"] else ""  # Convert to lowercase
    meal_types = prefs["meal_types"] if prefs["meal_types"] else set()  
    # Normalize meal_types by stripping spaces
    meal_types = {meal.strip() for meal in meal_types}
    # Generate meal plan
    meal_plan, final_docs = generate_meal_plan(
        vectorstore,
        user_avoid_ingredients,
        user_dislikes,
        query,
        user_likes,
        user_pref,
        size,
        protein_option,
        protein_category,
        meal_types
    )

    # Print the meal plan
    print(f"\nMeal Plan for User ID: {user_id}:\n{meal_plan}\n")
    time.sleep(4)  # Maintains RPM limit



Processing: user_data/processed_files/processed_66476957f69e6aba7d9ade81.json

Processing: user_data/processed_files/processed_64dde7c55d497c8a38be700d.json

Processing: user_data/processed_files/processed_651dae63bed63f9897fec329.json

Processing: user_data/processed_files/processed_67e19b0cb40e38cbe7284e88.json

Processing: user_data/processed_files/processed_67cc304e21dbdf828a2fa4e3.json

Processing: user_data/processed_files/processed_651e5e84bed63f9897009267.json

Processing: user_data/processed_files/processed_66e0a84abd17c8733adb739f.json

Processing: user_data/processed_files/processed_6620d7631b57fe5ac6cd6a2f.json

Processing: user_data/processed_files/processed_64ff5cafb78e129edca5bc95.json

Processing: user_data/processed_files/processed_671fade838819b8239f9046b.json

Processing: user_data/processed_files/processed_67c1adc382a8c3decbc249ba.json

Processing: user_data/processed_files/processed_67970d9a78f4171995f9726d.json

Processing: user_data/processed_files/processed_67a

In [15]:
user_preferences

{'66476957f69e6aba7d9ade81': {'user_pref': 'Balkan Mushroom Rice, Mac & Cheese with Cauliflower, Saloona with Green Beans , Thai Mango Salad, Maftoul with Zucchini & Minced Protein , Leek & Potato Fusilli Pasta. Highest Rated Dishes: Creamy Protein & Mash Potatoes, Mansaf , Fusilli Alfredo, Mac & Cheese with Cauliflower, Chipotle Lime Protein with Cauliflower Pilaf',
  'user_likes': 'Mediterranean cuisine',
  'user_avoid_ingredients': {'Liver'},
  'size': 'Large',
  'protein_option': '',
  'protein_category': 'low',
  'meal_types': {'dinner', 'lunch'},
  'query': 'Spice Level: Medium, Cuisine: Mediterranean. Popular Dishes: Balkan Mushroom Rice, Mac & Cheese with Cauliflower, Saloona with Green Beans , Thai Mango Salad, Maftoul with Zucchini & Minced Protein , Leek & Potato Fusilli Pasta. Highest Rated Dishes: Creamy Protein & Mash Potatoes, Mansaf , Fusilli Alfredo, Mac & Cheese with Cauliflower, Chipotle Lime Protein with Cauliflower Pilaf',
  'user_dislikes': ''},
 '64dde7c55d497c8a

#### json to csv

In [60]:
import json
import csv
import re
from collections import defaultdict

def process_mealplan_text(mealplan_text):
    # Split the text into individual user sections
    user_sections = re.split(r'Generating meal plan for User ID:', mealplan_text)[1:]
    
    all_user_data = []
    meal_types = set()
    
    for section in user_sections:
        # Extract user ID from the first line
        user_id = section.split('\n')[0].strip()
        
        # Find JSON part
        json_match = re.search(r'```json\n({.*?})\n```', section, re.DOTALL)
        if not json_match:
            continue
            
        try:
            meal_plan = json.loads(json_match.group(1))
            
            # Handle both list and dictionary formats for weeklyMealPlan
            if isinstance(meal_plan["weeklyMealPlan"], list):
                weekly_meals = {}
                for day_plan in meal_plan["weeklyMealPlan"]:
                    day_name = day_plan.get("day")
                    if day_name:
                        weekly_meals[day_name] = {k: v for k, v in day_plan.items() if k != "day"}
            else:
                weekly_meals = meal_plan["weeklyMealPlan"]
            
            # Collect all meal types across all users
            for day_meals in weekly_meals.values():
                if isinstance(day_meals, dict):
                    meal_types.update(day_meals.keys())
                
            # Format user data
            user_data = {"user_id": user_id}
            for day in ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]:
                if day in weekly_meals:
                    meals = []
                    for meal_type, meal_details in weekly_meals[day].items():
                        meals.append(f"{meal_type} - {meal_details}")
                    user_data[day] = ", ".join(meals)
                else:
                    user_data[day] = ""
            
            all_user_data.append(user_data)
            
        except (json.JSONDecodeError, KeyError) as e:
            print(f"Error processing user {user_id}: {str(e)}")
            continue
    
    return all_user_data, sorted(meal_types)

def save_to_csv(user_data, output_file="all_users_mealplans.csv"):
    fieldnames = ["user_id", "Monday", "Tuesday", "Wednesday", 
                 "Thursday", "Friday", "Saturday", "Sunday"]
    
    with open(output_file, 'w', newline='') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(user_data)
    
    print(f"Successfully saved meal plans for {len(user_data)} users to {output_file}")

# Example usage with a text file containing multiple users
def process_mealplan_file(input_file_path):
    with open(input_file_path, 'r') as file:
        mealplan_text = file.read()
    
    user_data, meal_types = process_mealplan_text(mealplan_text)
    save_to_csv(user_data)
    
    print(f"Detected meal types across all users: {', '.join(meal_types)}")

# Run the processor
process_mealplan_file("meal_plans.txt")  # Replace with your input file path

Error processing user 677a91bcd77f7aa7b009df89: 'weeklyMealPlan'
Error processing user 67c44ace191ae616ec296114: 'weeklyMealPlan'
Error processing user 673dcb9c8135a8501bf138f0: 'weeklyMealPlan'
Error processing user 6728f423792e409e6dbb29db: 'weeklyMealPlan'
Error processing user 66c9c9377555d575aebb37be: 'weeklyMealPlan'
Error processing user 67776acc850445ccebfade04: 'weeklyMealPlan'
Error processing user 64c8e979cf697b0d785e4038: 'weeklyMealPlan'
Error processing user 66d099c5d877ae126aa1d873: 'weeklyMealPlan'
Error processing user 662cab753c2816a43a179912: 'weeklyMealPlan'
Error processing user 67c45c88776c9d06c3f03beb: 'weeklyMealPlan'
Error processing user 67abadbfc8b635d2cccb0eca: 'weeklyMealPlan'
Error processing user 67aa6256c8b635d2ccae5077: 'weeklyMealPlan'
Error processing user 64bc013fc3b0d43faabe6d56: 'weeklyMealPlan'
Error processing user 64dd02a279e63ef3bb546192: 'weeklyMealPlan'
Successfully saved meal plans for 84 users to all_users_mealplans.csv
Detected meal types 

In [67]:
import json
import csv
import re
from collections import defaultdict

def process_mealplan_text(mealplan_text):
    user_sections = re.split(r'Generating meal plan for User ID:', mealplan_text)[1:]
    
    all_user_data = []
    meal_types = set()
    problem_users = []
    
    for section in user_sections:
        user_id = section.split('\n')[0].strip()
        
        json_match = re.search(r'```json\n({.*?})\n```', section, re.DOTALL)
        if not json_match:
            problem_users.append((user_id, "No JSON found"))
            continue
            
        try:
            meal_plan = json.loads(json_match.group(1))
            
            # Try different possible keys for the meal plan
            possible_keys = [
                'weeklyMealPlan',  # camelCase version
                'weekly_meal_plan',  # underscore version
                'mealPlan',
                'meal_plan',
                'plan',
                'weeklyPlan'
            ]
            
            weekly_meals = None
            for key in possible_keys:
                if key in meal_plan:
                    weekly_meals = meal_plan[key]
                    break
            
            if weekly_meals is None:
                problem_users.append((user_id, "No recognized meal plan key found"))
                continue
            
            # Convert list format to dictionary if needed
            if isinstance(weekly_meals, list):
                temp_meals = {}
                for item in weekly_meals:
                    if "day" in item:
                        temp_meals[item["day"]] = {k: v for k, v in item.items() if k != "day"}
                    elif "weekday" in item:  # Handle alternate day key
                        temp_meals[item["weekday"]] = {k: v for k, v in item.items() if k != "weekday"}
                weekly_meals = temp_meals
            
            # Ensure we have a dictionary at this point
            if not isinstance(weekly_meals, dict):
                problem_users.append((user_id, f"Unexpected meal plan format: {type(weekly_meals)}"))
                continue
            
            # Collect meal types and format data
            current_meal_types = set()
            user_data = {"user_id": user_id}
            
            for day in ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]:
                if day in weekly_meals:
                    day_meals = weekly_meals[day]
                    if isinstance(day_meals, dict):
                        meals = []
                        for meal_type, meal_details in day_meals.items():
                            meals.append(f"{meal_type} - {meal_details}")
                            current_meal_types.add(meal_type)
                        user_data[day] = ", ".join(meals)
                    else:
                        user_data[day] = str(day_meals)  # Fallback for non-dict formats
                else:
                    user_data[day] = ""
            
            meal_types.update(current_meal_types)
            all_user_data.append(user_data)
            
        except Exception as e:
            problem_users.append((user_id, f"Processing error: {str(e)}"))
            continue
    
    # Print summary of problematic users
    if problem_users:
        print("\nProblematic users:")
        for user_id, reason in problem_users:
            print(f"- {user_id}: {reason}")
    else:
        print("\nAll users processed successfully!")
    
    return all_user_data, sorted(meal_types)

def save_to_csv(user_data, output_file="test.csv"):
    fieldnames = ["user_id", "Monday", "Tuesday", "Wednesday", 
                 "Thursday", "Friday", "Saturday", "Sunday"]
    
    with open(output_file, 'w', newline='') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(user_data)
    
    print(f"\nSuccessfully saved meal plans for {len(user_data)} users to {output_file}")

def process_mealplan_file(input_file_path):
    with open(input_file_path, 'r') as file:
        mealplan_text = file.read()
    
    user_data, meal_types = process_mealplan_text(mealplan_text)
    save_to_csv(user_data)
    
    print(f"\nDetected meal types across all users: {', '.join(meal_types)}")

# Run the processor
process_mealplan_file("test.txt")


All users processed successfully!

Successfully saved meal plans for 10 users to test.csv

Detected meal types across all users: 


In [14]:
import json
import csv
import re

def extract_user_plans(text):
    """Extract all user plans from the text"""
    user_sections = re.split(r'Generating meal plan for User ID:', text)[1:]
    user_plans = []
    
    for section in user_sections:
        user_id = section.split('\n')[0].strip()
        json_match = re.search(r'```json\n({.*?})\n```', section, re.DOTALL)
        
        if not json_match:
            print(f"Skipping user {user_id} - no JSON found")
            continue
            
        try:
            plan_data = json.loads(json_match.group(1))
            user_plans.append((user_id, plan_data))
        except json.JSONDecodeError as e:
            print(f"Skipping user {user_id} - invalid JSON: {str(e)}")
            
    return user_plans

def format_day_meals(day_meals):
    """Format all meals for a day into the required string format"""
    if not isinstance(day_meals, dict):
        return ""
    
    formatted = []
    for meal_type, meal_details in day_meals.items():
        formatted.append(f"{meal_type} - {meal_details}")
    return ", ".join(formatted)

def create_csv_output(user_plans, output_file):
    """Create the final CSV output"""
    fieldnames = ["user_id", "Monday", "Tuesday", "Wednesday", 
                 "Thursday", "Friday", "Saturday", "Sunday"]
    
    with open(output_file, 'w', newline='') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        
        for user_id, plan_data in user_plans:
            # Plan data is already in the correct format (days as top-level keys)
            row = {"user_id": user_id}
            for day in fieldnames[1:]:  # Skip user_id column
                row[day] = format_day_meals(plan_data.get(day, {}))
            
            writer.writerow(row)

def process_mealplan_file(input_file, output_file):
    """Main processing function"""
    with open(input_file, 'r') as file:
        text = file.read()
    
    user_plans = extract_user_plans(text)
    create_csv_output(user_plans, output_file)
    print(f"\nSuccessfully processed {len(user_plans)} users to {output_file}")

# Example usage:
process_mealplan_file("meal_plans.txt", "meal_plans.csv")


Successfully processed 99 users to meal_plans.csv
